In [1]:
import os
import cv2
import numpy as np
from skimage import io, color, restoration, img_as_ubyte
import matplotlib.pyplot as plt

In [ ]:
# 禁用弹出窗口
plt.ioff()

# 设置主目录路径（请替换为你自己的主目录路径）
root_dir = ''  # ←←← 替换成你的路径

In [ ]:
# 遍历主目录下的所有“subsub 文件夹”
for root, dirs, files in os.walk(root_dir):
    # 如果当前目录下包含指定三张图，说明是目标目录
    if all(name in files for name in ['参考物.png', '参考散斑.png', '未知物散斑.png']):
        print(f'处理中：{root}')

        try:
            # ========== 图像读取 ==========
            R = io.imread(os.path.join(root, '参考物.png'))
            R_D = io.imread(os.path.join(root, '参考散斑.png'))
            U_D = io.imread(os.path.join(root, '未知物散斑.png'))

            # ========== 灰度归一化 ==========
            R = color.rgb2gray(R)
            R_D = color.rgb2gray(R_D)
            U_D = color.rgb2gray(U_D)

            R_D_np = img_as_ubyte(R_D)
            U_np = img_as_ubyte(U_D)

            # ========== PSF估计 ==========
            PSF, _ = restoration.unsupervised_wiener(R_D, R)
            plt.figure()
            plt.imshow(np.abs(PSF), cmap='hot')
            plt.axis('off')
            plt.savefig(os.path.join(root, '点扩散函数_PSF.png'))
            plt.close()

            # ========== 维纳滤波恢复 ==========
            U_restored, _ = restoration.unsupervised_wiener(U_D, PSF)
            plt.figure()
            plt.imshow(U_restored, cmap='hot')
            plt.axis('equal')
            plt.savefig(os.path.join(root, '恢复图像_维纳滤波.png'))
            plt.close()

            # ========== 阈值分割 ==========
            threshold = 0.25
            U_th = np.clip(U_restored, 0, 1)
            U_th[U_th > threshold] = 1
            plt.figure()
            plt.imshow(U_th, cmap='hot')
            plt.axis('equal')
            plt.savefig(os.path.join(root, '阈值分割图像.png'))
            plt.close()

            # ========== 互相关系数计算 ==========
            cross_corr = cv2.matchTemplate(R_D_np, U_np, cv2.TM_CCOEFF_NORMED)
            plt.figure()
            plt.imshow(cross_corr, cmap='hot', vmin=-1, vmax=1)
            plt.axis('off')
            plt.savefig(os.path.join(root, '互相关系数图.png'))
            plt.close()

        except Exception as e:
            print(f'跳过 {root}，发生错误：{e}')

In [ ]:
for root, dirs, files in os.walk(root_dir):
    if all(name in files for name in ['参考物.png', '参考散斑.png', '未知物散斑.png']):
        print(f'处理中：{root}')

        try:
            # === 图像读取 ===
            R = io.imread(os.path.join(root, '参考物.png'))
            R_D = io.imread(os.path.join(root, '参考散斑.png'))
            U_D = io.imread(os.path.join(root, '未知物散斑.png'))

            # === 灰度归一化 ===
            R = color.rgb2gray(R)
            R_D = color.rgb2gray(R_D)
            U_D = color.rgb2gray(U_D)

            R_D_np = img_as_ubyte(R_D)
            U_np = img_as_ubyte(U_D)

            # === PSF估计 ===
            PSF, _ = restoration.unsupervised_wiener(R_D, R)

            # === 维纳滤波恢复 ===
            U_restored, _ = restoration.unsupervised_wiener(U_D, PSF)

            # === 阈值分割 ===
            threshold = 0.25
            U_th = np.clip(U_restored, 0, 1)
            U_th[U_th > threshold] = 1

            # === 拼图输出（直接用内存图像） ===
            def crop_center(img):
                h, w = img.shape[:2]
                h_start, h_end = h // 4, 3 * h // 4
                w_start, w_end = w // 4, 3 * w // 4
                return img[h_start:h_end, w_start:w_end]

            imgs = [U_D, np.abs(PSF), U_th]
            titles = ['Unknown Speckle', 'PSF', 'Reconstructed']

            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            for ax, img, title in zip(axes, imgs, titles):
                ax.imshow(crop_center(img), cmap='hot')
                ax.set_title(title)
                ax.set_xticks([])
                ax.set_yticks([])

            plt.tight_layout()
            plt.savefig(os.path.join(root, '图像对比图.png'))
            plt.savefig(os.path.join(root, '图像对比图.pdf'))
            plt.close()

        except Exception as e:
            print(f'跳过 {root}，发生错误：{e}')